In [8]:
# -----------------------------------------------------------------------------
# 0. 프로젝트 폴더와 환경변수 준비
# -----------------------------------------------------------------------------
#
# 이 단계에서는 파일 경로와 .env만 필요합니다.
# 따라서 Path와 load_dotenv만 불러옵니다.
# -----------------------------------------------------------------------------

from pathlib import Path
from dotenv import load_dotenv

# 현재 Notebook을 실행하는 폴더를 프로젝트 기준 폴더로 사용합니다.
PROJECT_DIR = Path(".")

# news 원문 저장할 폴더입니다.
RAW_DIR = Path("data") / "news"

# 상위 폴더가 없어도 함께 만들고, 이미 있어도 오류를 내지 않습니다.
RAW_DIR.mkdir(parents=True, exist_ok=True)


# DB 비밀번호, API Key, Hugging Face Token 등을 환경변수로 읽습니다.
load_dotenv(".env")

print("현재 폴더:", Path.cwd())
print("news 원문 폴더:", RAW_DIR)

현재 폴더: /Users/minseok_geum/Desktop/SKKUFI/FDS/ai_campus_mini_proj
news 원문 폴더: data/news


In [15]:
import requests
import pandas as pd
import re
from html import unescape
from datetime import datetime
from email.utils import parsedate_to_datetime
from zoneinfo import ZoneInfo
import os



# =========================================================
# 1. NAVER API HUB 인증 정보
# =========================================================

NAVER_CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
NAVER_CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")
API_URL = "https://naverapihub.apigw.ntruss.com/search/v1/news"


# =========================================================
# 2. HTML 태그 제거
# =========================================================

def clean_text(text):
    """
    뉴스 검색 결과의 <b>...</b> 등의 HTML 태그 제거
    """
    if not text:
        return ""

    text = re.sub(r"<[^>]+>", "", text)
    return unescape(text)


# =========================================================
# 3. 특정 기업 + 특정 날짜 뉴스 검색
# =========================================================

def search_company_news(company, target_date):
    """
    Parameters
    ----------
    company : str
        검색할 기업명
        ex) "삼성전자"

    target_date : str
        검색할 날짜
        형식: YYYY-MM-DD
        ex) "2026-10-01"

    Returns
    -------
    pandas.DataFrame
        해당 날짜에 검색된 뉴스 목록
    """

    target_date = datetime.strptime(
        target_date,
        "%Y-%m-%d"
    ).date()

    headers = {
        "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
        "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET
    }

    results = []

    # 검색 API는 start 최대값이 1000이므로
    # 100개씩 최대 10번 요청
    for start in range(1, 1001, 100):

        params = {
            "query": company,
            "display": 100,
            "start": start,
            "sort": "date",
            "format": "json"
        }

        response = requests.get(
            API_URL,
            headers=headers,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        items = data.get("items", [])

        if not items:
            break

        reached_older_date = False

        for item in items:

            # 예:
            # Tue, 06 Oct 2026 09:21:00 +0900
            pub_datetime = parsedate_to_datetime(
                item["pubDate"]
            )

            # 한국 시간 기준 날짜로 변환
            pub_datetime = pub_datetime.astimezone(
                ZoneInfo("Asia/Seoul")
            )

            pub_date = pub_datetime.date()

            # 원하는 날짜 기사
            if pub_date == target_date:

                results.append({
                    "company": company,
                    "title": clean_text(item.get("title", "")),
                    "description": clean_text(
                        item.get("description", "")
                    ),
                    "pub_datetime": pub_datetime,
                    "original_link": item.get(
                        "originallink",
                        ""
                    ),
                    "naver_link": item.get(
                        "link",
                        ""
                    )
                })

            # target_date보다 과거 기사까지 내려갔다면
            # 이후 결과도 더 오래된 기사이므로 검색 종료
            elif pub_date < target_date:
                reached_older_date = True
                break

        if reached_older_date:
            break

    df = pd.DataFrame(results)

    if not df.empty:
        df = df.sort_values(
            "pub_datetime",
            ascending=False
        ).reset_index(drop=True)

    return df

In [20]:
df = search_company_news("삼성전자 HBM", "2026-10-04")
df

,company,title,description,pub_datetime,original_link,naver_link
0,삼성전자 HBM,삼성 HBM4 수율 80% 반전···내년 상반기 SK하이닉스 잡나,삼성전자 HBM4 수율이 80% 안팎까지 올라온 배경에는 HBM의 재료가 되는 10...,2026-10-04 22:02:00+09:00,https://www.womaneconomy.co.kr/news/articleVie...,https://www.womaneconomy.co.kr/news/articleVie...
1,삼성전자 HBM,마이크론이 먼저 확인한 메모리 호황…삼성·SK하이닉스는 '비용 변수',삼성전자는 지난 2월 현재 세대 제품인 HBM4(6세대) 양산 출하를 시작한 뒤 공...,2026-10-04 20:41:00+09:00,http://www.inews24.com/view/2011749,https://n.news.naver.com/mnews/article/031/000...
2,삼성전자 HBM,"[연합뉴스TV 스페셜] 443회 : AI패권전쟁, K-반도체의 미래는?",PD 명노현 AD 최병철 구성 이은혜 유영지 #AI패권전쟁 #K반도체 #반도체 #슈...,2026-10-04 20:01:00+09:00,http://www.yonhapnewstv.co.kr/news/MYH20261002...,https://n.news.naver.com/mnews/article/422/000...
3,삼성전자 HBM,"[현장] ""아산에 오면 삼성과 함께 큰다""…113조 투자 품고 '기업도시 대...","삼성디스플레이가 67조원, 삼성전자가 HBM 등 반도체 후공정 분야에 46조원을 투...",2026-10-04 19:08:00+09:00,https://weekly.hankooki.com/news/articleView.h...,https://weekly.hankooki.com/news/articleView.h...
4,삼성전자 HBM,"""삼전닉스, 年 1000조 번다"" 전망에… 추가 주주환원 기대",삼성전자의 HBM4 가격이 업계 최고 수준으로 책정될 것이라는 예상에서다. 트렌스포...,2026-10-04 18:35:00+09:00,https://www.fnnews.com/news/202610041834133237,https://n.news.naver.com/mnews/article/014/000...
5,삼성전자 HBM,3분기 영업익 764%↑…삼성전자 ‘100조 클럽’,실제 삼성전자가 3·4분기부터 HBM4 양산을 본격화하면서 HBM 매출이 직전 분기...,2026-10-04 18:27:00+09:00,https://www.fnnews.com/news/202610041826049921,https://n.news.naver.com/mnews/article/014/000...
6,삼성전자 HBM,어플라이드·키옥시아 EPIC 센터 협력…AI 메모리 공동 R&D 협력,삼성전자가 지난 2월 13일 합류를 발표했다. 마이크론은 지난 3월 11일 창립 파...,2026-10-04 18:16:00+09:00,https://www.econovill.com/news/articleView.htm...,https://www.econovill.com/news/articleView.htm...
7,삼성전자 HBM,"삼성전자, 사상 첫 '분기 영업익 100조' 눈앞… 메모리 '슈퍼 호황' 불붙...",삼성전자는 HBM과 범용 메모리 양쪽에서 수혜를 볼 수 있는 업체다. 회사는 2분기...,2026-10-04 18:00:00+09:00,https://www.financialpost.co.kr/news/articleVi...,https://www.financialpost.co.kr/news/articleVi...
8,삼성전자 HBM,"마이크론 투자 확대에도…""메모리 공급과잉 우려는 시기상조""",지난 1일 국내 증시에서 삼성전자와 SK하이닉스는 각각 0.7%대 하락세로 거래를 ...,2026-10-04 18:00:00+09:00,https://www.ebn.co.kr/news/articleView.html?id...,https://www.ebn.co.kr/news/articleView.html?id...
9,삼성전자 HBM,국가핵심기술 10건 중 9건 中으로 샜다,"고대역폭메모리(HBM) 공정 기술 같은 민감한 내부 정보를 요구한다""며 ""끓는 물에...",2026-10-04 17:33:00+09:00,https://www.mk.co.kr/article/12168152,https://n.news.naver.com/mnews/article/009/000...


In [6]:
print(df.loc[0, 'description'])


이재용 삼성전자 회장은 59조1878억원에서 45조9894억원으로 13조1984억원 감소했다. 최태원 SK그룹 회장은 3조1862억원, 정의선 현대차그룹 회장은 1조1813억원 줄었다. 세 사람의 주식재산 감소액만 17조5659억원에... 
